# Convergence and Robustness of the Pathway Classification

Checks that the primary-label classification produced by
`01_pathway_classification.ipynb` is stable and not an artifact of the
number of sampled pathways or of the exact quantile thresholds used. All
computation lives in `gbctp.reliability`; this notebook only configures
inputs, runs the checks, and plots the results.

Produces: class fingerprints, subsampling convergence (main text),
threshold convergence, bootstrap uncertainty, and threshold sensitivity (SI).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gbctp.reliability import (
    DEFAULT_LABEL_ORDER,
    classify_dataframe,
    collect_profiles_for_label,
    ordered_present_labels,
    run_subsampling_convergence,
    run_threshold_sensitivity,
    summarize_bootstrap,
)

# ---------------------------------------------------------------------------
# INPUT DATA
#
# ORDER_DESCRIPTOR_FILE / PRIMARY_LABEL_FILE are the
# paths_order_descriptors_with_archetype_labels.csv / paths_with_primary_labels.csv
# files written by 01_pathway_classification.ipynb (gbctp.pipeline.run_pipeline).
# REPRESENTATIVE_FILE (optional) is that notebook's representative_paths_for_dynamics.csv.
#
# Class-fingerprint plots additionally need the per-pathway
# "<name>.JCE_vectors.csv" sidecar files next to the original coupling CSVs
# (written automatically by run_pipeline when save_per_dimer_vectors=True).
# ---------------------------------------------------------------------------
ROOT_DIR = Path("<path/to/output_dir>")  # TODO: same OUTPUT_DIR as notebook 01
PRIMARY_LABEL_FILE = ROOT_DIR / "paths_with_primary_labels.csv"
ORDER_DESCRIPTOR_FILE = ROOT_DIR / "paths_order_descriptors_with_archetype_labels.csv"
REPRESENTATIVE_FILE = ROOT_DIR / "representative_paths_for_dynamics.csv"

# Optional per-pathway or per-class mobility file. Leave as None if unavailable.
# Accepted formats: columns [path, mobility] or [label_primary, mobility].
MOBILITY_FILE = None
# MOBILITY_FILE = ROOT_DIR / "pathway_mobility.csv"

OUTPUT_DIR = ROOT_DIR / "classification_reliability_figures"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 123
N_BOOT = 300
N_MIN = 30
N_STEP = 25

GRID_SIZE = 100
NORMALIZATION = "median"  # one of: "raw", "median", "max", "first_last"

# If your GB is approximately in the middle of the normalized path:
GB_REGION = (0.40, 0.60)

LABEL_ORDER = DEFAULT_LABEL_ORDER


## Plot helpers

In [ ]:
def savefig(fig, name):
    png, pdf = OUTPUT_DIR / f"{name}.png", OUTPUT_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved:\n  {png}\n  {pdf}")


## Class fingerprints

Median (with IQR) of the normalized effective-link profile E_i = J_i C_i
along the pathway, grouped by primary label.

In [ ]:
def plot_class_fingerprints():
    df_primary = pd.read_csv(PRIMARY_LABEL_FILE)
    labels = ordered_present_labels(df_primary["label_primary"], LABEL_ORDER)
    grid = np.linspace(0.0, 1.0, GRID_SIZE)

    fig, axes = plt.subplots(len(labels), 1, figsize=(7.2, 2.6 * len(labels)), sharex=True)
    if len(labels) == 1:
        axes = [axes]

    rows = []
    for ax, label in zip(axes, labels):
        profiles = collect_profiles_for_label(df_primary, label, grid, normalization=NORMALIZATION)
        if profiles is None:
            ax.text(0.5, 0.5, f"No profiles for {label}", ha="center", va="center")
            continue

        med = np.median(profiles, axis=0)
        q25 = np.percentile(profiles, 25, axis=0)
        q75 = np.percentile(profiles, 75, axis=0)

        ax.axvspan(GB_REGION[0], GB_REGION[1], alpha=0.12, label="GB region")
        ax.plot(grid, med, linewidth=2.3, label="Median")
        ax.fill_between(grid, q25, q75, alpha=0.25, label="IQR")

        ax.set_title(f"{label}  (N = {profiles.shape[0]})")
        ax.set_ylabel(r"$E_i / \mathrm{median}(E_i)$" if NORMALIZATION == "median" else r"$E_i$")
        ax.grid(alpha=0.3)
        ax.legend(frameon=False, fontsize=8)

        for x, m, lo, hi in zip(grid, med, q25, q75):
            rows.append({"label_primary": label, "x_norm": x, "median": m, "q25": lo, "q75": hi,
                         "n_profiles": profiles.shape[0], "normalization": NORMALIZATION})

    axes[-1].set_xlabel("Normalized pathway position")
    fig.suptitle(r"Class-wise fingerprints of effective link strength $E_i = J_i C_i$", fontsize=13, y=1.01)
    plt.tight_layout()

    out_csv = OUTPUT_DIR / "MAIN_class_fingerprint_E_profiles_data.csv"
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"Saved data: {out_csv}")
    savefig(fig, "MAIN_class_fingerprint_E_profiles")
    plt.show()


plot_class_fingerprints()


## Subsampling convergence

Repeatedly re-classifies random subsets of the sampled pathways
(recomputing quantile thresholds each time) to test whether primary-label
class fractions have stabilized with respect to the number of sampled
pathways.

In [ ]:
df_all = pd.read_csv(ORDER_DESCRIPTOR_FILE)

df_conv, df_full, labels, full_frac, full_gb_frac = run_subsampling_convergence(
    df_all, n_min=N_MIN, n_step=N_STEP, n_boot=N_BOOT, random_seed=RANDOM_SEED, label_order=LABEL_ORDER,
)
df_conv.to_csv(OUTPUT_DIR / "subsampling_convergence_results.csv", index=False)

print("Full dataset label counts:")
print(df_full["label_primary"].value_counts())


In [ ]:
def plot_main_subsampling_convergence(df_conv, labels, full_frac, full_gb_frac):
    summary_primary = summarize_bootstrap(df_conv, "primary_fraction")
    summary_gb = summarize_bootstrap(df_conv, "GB_fraction")

    fig, axes = plt.subplots(2, 1, figsize=(7.2, 7.2))

    ax = axes[0]
    for lab in labels:
        sub = summary_primary[summary_primary["label"] == lab]
        if len(sub) == 0:
            continue
        ax.plot(sub["n_sample"], sub["mean"], linewidth=2, label=lab)
        ax.fill_between(sub["n_sample"], sub["q025"], sub["q975"], alpha=0.20)
        if lab in full_frac.index:
            ax.axhline(full_frac[lab], linestyle="--", linewidth=1, alpha=0.35)
    ax.set_title("Subsampling convergence of primary pathway classes")
    ax.set_ylabel("Class fraction")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8, ncol=2)

    ax = axes[1]
    for lab in ["GB-crossing (recovering)", "GB-blocked (non-recovering)"]:
        sub = summary_gb[summary_gb["label"] == lab]
        if len(sub) == 0:
            continue
        ax.plot(sub["n_sample"], sub["mean"], linewidth=2, label=lab)
        ax.fill_between(sub["n_sample"], sub["q025"], sub["q975"], alpha=0.20)
        if lab in full_gb_frac.index:
            ax.axhline(full_gb_frac[lab], linestyle="--", linewidth=1, alpha=0.35)
    ax.set_title("Subsampling convergence of GB recovery behavior")
    ax.set_xlabel("Number of sampled pathways")
    ax.set_ylabel("Fraction within GB-limited pathways")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8)

    plt.tight_layout()
    savefig(fig, "MAIN_subsampling_convergence")
    plt.show()


plot_main_subsampling_convergence(df_conv, labels, full_frac, full_gb_frac)


## Class-weighted mobility convergence (optional)

Only runs if a mobility CSV is supplied via `MOBILITY_FILE` above.

In [ ]:
def plot_weighted_mobility_convergence(df_conv):
    """Supports two mobility file formats: [path, mobility] or [label_primary, mobility]."""
    if MOBILITY_FILE is None:
        print("Skipping weighted mobility convergence: MOBILITY_FILE = None")
        return

    mob = pd.read_csv(MOBILITY_FILE)
    rng = np.random.default_rng(RANDOM_SEED)
    n_total = len(df_all)
    n_values = sorted(df_conv["n_sample"].unique())
    rows = []

    if {"path", "mobility"}.issubset(mob.columns):
        df_all_with_mob = df_all.merge(mob[["path", "mobility"]], on="path", how="left")
        for n_sample in n_values:
            for boot in range(N_BOOT):
                idx = rng.choice(n_total, size=n_sample, replace=False)
                df_sub_lab, _ = classify_dataframe(df_all_with_mob.iloc[idx])
                valid = df_sub_lab["mobility"].notna()
                if valid.sum() == 0:
                    continue
                rows.append({"n_sample": n_sample, "bootstrap": boot,
                             "weighted_mobility": df_sub_lab.loc[valid, "mobility"].mean()})

    elif {"label_primary", "mobility"}.issubset(mob.columns):
        class_mob = dict(zip(mob["label_primary"], mob["mobility"]))
        for n_sample in n_values:
            for boot in range(N_BOOT):
                idx = rng.choice(n_total, size=n_sample, replace=False)
                df_sub_lab, _ = classify_dataframe(df_all.iloc[idx])
                frac = df_sub_lab["label_primary"].value_counts(normalize=True)
                weighted_mob = sum(f * class_mob[label] for label, f in frac.items() if label in class_mob)
                rows.append({"n_sample": n_sample, "bootstrap": boot, "weighted_mobility": weighted_mob})
    else:
        raise ValueError("MOBILITY_FILE must have columns [path, mobility] or [label_primary, mobility]")

    df_mob_conv = pd.DataFrame(rows)
    if len(df_mob_conv) == 0:
        print("No mobility convergence data generated.")
        return

    df_mob_conv.to_csv(OUTPUT_DIR / "weighted_mobility_convergence_results.csv", index=False)
    summary = df_mob_conv.groupby("n_sample")["weighted_mobility"].agg(
        mean="mean", q025=lambda x: np.percentile(x, 2.5), q975=lambda x: np.percentile(x, 97.5),
    ).reset_index()

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(summary["n_sample"], summary["mean"], linewidth=2)
    ax.fill_between(summary["n_sample"], summary["q025"], summary["q975"], alpha=0.25)
    ax.set_xlabel("Number of sampled pathways")
    ax.set_ylabel("Class-weighted mobility")
    ax.set_title("Convergence of class-weighted mobility")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    savefig(fig, "MAIN_weighted_mobility_convergence")
    plt.show()


plot_weighted_mobility_convergence(df_conv)


## Threshold convergence (SI)

Convergence of the quantile-based classification thresholds themselves as
more pathways are included.

In [ ]:
def plot_threshold_convergence(df_conv):
    summary = summarize_bootstrap(df_conv, "threshold")
    threshold_labels = [lab for lab in summary["label"].unique()]

    fig, axes = plt.subplots(len(threshold_labels), 1, figsize=(7.2, 2.0 * len(threshold_labels)), sharex=True)
    if len(threshold_labels) == 1:
        axes = [axes]

    for ax, lab in zip(axes, threshold_labels):
        sub = summary[summary["label"] == lab]
        ax.plot(sub["n_sample"], sub["mean"], linewidth=2)
        ax.fill_between(sub["n_sample"], sub["q025"], sub["q975"], alpha=0.25)
        ax.set_ylabel(lab)
        ax.grid(alpha=0.3)

    axes[-1].set_xlabel("Number of sampled pathways")
    fig.suptitle("Convergence of quantile-based classification thresholds", y=1.01)
    plt.tight_layout()
    savefig(fig, "SI_threshold_convergence")
    plt.show()


plot_threshold_convergence(df_conv)


## Bootstrap uncertainty (SI)

95% bootstrap interval width of each class fraction as a function of sample size.

In [ ]:
def plot_bootstrap_uncertainty(df_conv, labels):
    summary = summarize_bootstrap(df_conv, "primary_fraction")

    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    for lab in labels:
        sub = summary[summary["label"] == lab]
        if len(sub) == 0:
            continue
        ax.plot(sub["n_sample"], sub["q975"] - sub["q025"], linewidth=2, label=lab)

    ax.set_xlabel("Number of sampled pathways")
    ax.set_ylabel("95% bootstrap interval width")
    ax.set_title("Bootstrap uncertainty of class fractions")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    savefig(fig, "SI_bootstrap_label_uncertainty")
    plt.show()


plot_bootstrap_uncertainty(df_conv, labels)


## Threshold sensitivity (SI)

Re-classifies the full dataset under low/default/high quantile cutoffs and
reports label agreement with the default classification.

In [ ]:
df_sens, df_agree = run_threshold_sensitivity(df_all)
df_sens.to_csv(OUTPUT_DIR / "threshold_sensitivity_class_fractions.csv", index=False)
df_agree.to_csv(OUTPUT_DIR / "threshold_sensitivity_label_agreement.csv", index=False)


def plot_threshold_sensitivity(df_sens, df_agree):
    labels = ordered_present_labels(df_sens["label_primary"], LABEL_ORDER)
    settings = list(df_sens["setting"].unique())
    x = np.arange(len(settings))
    width = 0.8 / max(1, len(labels))

    fig, axes = plt.subplots(2, 1, figsize=(7.2, 7.0))

    ax = axes[0]
    for i, lab in enumerate(labels):
        vals = [
            float(df_sens.loc[(df_sens["setting"] == s) & (df_sens["label_primary"] == lab), "fraction"].iloc[0])
            if len(df_sens[(df_sens["setting"] == s) & (df_sens["label_primary"] == lab)]) > 0 else 0.0
            for s in settings
        ]
        ax.bar(x + (i - len(labels) / 2) * width + width / 2, vals, width=width, label=lab)
    ax.set_xticks(x)
    ax.set_xticklabels(settings)
    ax.set_ylabel("Class fraction")
    ax.set_title("Sensitivity of class fractions to quantile thresholds")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(frameon=False, fontsize=8, ncol=2)

    ax = axes[1]
    ax.bar(df_agree["setting"], df_agree["agreement_with_default"], width=0.6)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Agreement with default labels")
    ax.set_title("Path-level label agreement under threshold perturbation")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    savefig(fig, "SI_threshold_sensitivity")
    plt.show()


plot_threshold_sensitivity(df_sens, df_agree)
